# Transformers, Day 2: Finishing the Block, Then Running It
**COMP 395 — Deep Learning**

Everything starts from your **Assignment #4** sheet: the words `not very good` and your $W_Q$, $W_K$, $W_V$.

| Part | Slide | What you see |
|---|---|---|
| 1 | Feed Forward | Attention mixes words; Feed Forward doesn't |
| 2 | LayerNorm | Every row comes out the same size |
| 3 | KV cache | The cache filling up, one word at a time |
| 4 | How big is the cache? | Real models, real gigabytes |

**All the code is written for you.** Run the cells in order. Cells marked **Try** ask you to change one thing and run again.

In [ ]:
import time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.set_printoptions(precision=2, sci_mode=False)

# Your Assignment #4 sheet
words = ["not", "very", "good"]
X   = torch.tensor([[-1., 1], [-2, 1], [2, -1]])      # one row per word
W_Q = torch.tensor([[-1., -1], [0, 1]])
W_K = torch.tensor([[-1., 0], [-1, -1]])
W_V = torch.tensor([[0., -1], [-1, 1]])

def attention(X):
    '''Assignment #4: every word looks at every word.'''
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    return torch.softmax(Q @ K.T, dim=-1) @ V

torch.manual_seed(0)
feed_forward = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))   # one small MLP, used for every word

show = lambda row: "[" + ", ".join(f"{v:5.2f}" for v in row.tolist()) + "]"
print("ready")

---
## Part 1 — Attention mixes words; Feed Forward doesn't (paper §3.3)

We change **only** `good`'s embedding, then check every word's output from attention and from Feed Forward.

**Try:** change a different word instead (`changed = "not"`).

In [ ]:
changed = "good"                           # <- which word to change
X2 = X.clone()
X2[words.index(changed)] += torch.tensor([0., 2.])

with torch.no_grad():
    att_before, att_after = attention(X), attention(X2)
    ff_before,  ff_after  = feed_forward(X), feed_forward(X2)

print(f"Changed only '{changed}'. Whose output moved?\n")
print("word    attention    feed forward")
for i, w in enumerate(words):
    a = "moved" if not torch.allclose(att_before[i], att_after[i]) else "same"
    f = "moved" if not torch.allclose(ff_before[i],  ff_after[i])  else "same"
    print(f"{w:6s}  {a:11s}  {f}")

**Question 1** — Why does changing one word move **every** attention output, but only **one** Feed Forward output?

*Your answer:*

---
## Part 2 — LayerNorm

Each word's row is rescaled to mean 0, spread 1.

**Try:** put your own numbers in the rows.

In [ ]:
rows = torch.tensor([[2., 4., 6.],
                     [200., 400., 600.]])     # <- try your own
print(nn.functional.layer_norm(rows, (3,)))

**Question 2** — Both rows came out the same. What does LayerNorm keep, and what does it throw away?

*Your answer:*

---
## Part 3 — The KV cache, one word at a time

Pretend the decoder is writing `not very good` one word at a time. When it writes a word, that word may look at itself and every word **before** it (the mask). With a cache:

1. compute $k$ and $v$ for the **new word only**,
2. add them as a new row of $K$ and $V$,
3. compare the new word's query with **every** row of $K$, and mix **every** row of $V$.

Watch $K$ grow.

In [ ]:
K_cache, V_cache = [], []

for t, w in enumerate(words):
    x = X[t]
    K_cache.append(x @ W_K)                   # step 1-2: only the new word's k and v
    V_cache.append(x @ W_V)
    K, V = torch.stack(K_cache), torch.stack(V_cache)
    q = x @ W_Q
    weights = torch.softmax(K @ q, dim=0)     # step 3: compare with every stored key

    print(f"--- writing '{w}' ---")
    print("K cache (one row per word so far):")
    for j in range(len(K_cache)):
        print(f"   {words[j]:5s} {show(K[j])}")
    print("attention weights:", show(weights), "\n")

**Question 3** — When writing `good`, how many rows of $K$ did we **compute**, and how many did we **use**? Do the weights for `good` match its row of $A$ on your Assignment #4 sheet? Why?

*Your answer:*

### Counting the work

Without a cache, writing word $t$ means recomputing $k, v$ for all $t$ words so far. With a cache, it's 1 new row each time. This is the triangle vs. the diagonal on the slide.

**Try:** add a bigger length to `lengths`.

In [ ]:
lengths = [3, 10, 100, 1000]                 # <- words written
print(" words     rows computed, no cache     rows computed, KV cache")
for n in lengths:
    no_cache = sum(range(1, n + 1))           # 1 + 2 + ... + n
    print(f"{n:6d}     {no_cache:22,d}     {n:20,d}")

### Timing it

Now with real sizes: 256 numbers per word. We don't have a trained model, so random vectors stand in for words. That's fine: we're only measuring how much **work** each method does, and both produce identical results.

In [ ]:
d = 256
torch.manual_seed(1)
Wq, Wk, Wv = (torch.randn(d, d) / d**0.5 for _ in range(3))

def generate(n, use_cache):
    torch.manual_seed(2)
    X = torch.zeros(n + 1, d); X[0] = torch.randn(d)            # the [START] token
    K = torch.zeros(n + 1, d); V = torch.zeros(n + 1, d)        # the cache
    for t in range(n):
        q = X[t] @ Wq
        if use_cache:                                           # 1 new row
            K[t], V[t] = X[t] @ Wk, X[t] @ Wv
            Kt, Vt = K[:t + 1], V[:t + 1]
        else:                                                   # all t + 1 rows, again
            Kt, Vt = X[:t + 1] @ Wk, X[:t + 1] @ Wv
        X[t + 1] = torch.tanh(torch.softmax(Kt @ q / d**0.5, dim=0) @ Vt)   # next "word"
    return X

lengths = [200, 400, 800, 1200, 1600]
no, yes = [], []
for n in lengths:
    t0 = time.time(); a = generate(n, False); no.append(time.time() - t0)
    t0 = time.time(); b = generate(n, True);  yes.append(time.time() - t0)
print("identical results:", torch.allclose(a, b, atol=1e-4))

plt.plot(lengths, no, "o-", label="no cache (the triangle)")
plt.plot(lengths, yes, "o-", label="KV cache (the diagonal)")
plt.xlabel("words written"); plt.ylabel("seconds"); plt.legend(); plt.grid(alpha=0.3); plt.show()

**Question 4** — The results are identical. So what does the cache cost, and what does it buy?

*Your answer:*

---
## Part 4 — How big does the cache get?

Bytes $= 2\ (K, V) \times$ layers $\times$ numbers per word $\times$ words $\times$ 2 bytes per number.

**Try:** change `words_so_far`.

In [ ]:
def kv_cache_gb(layers, d_model, n_words, bytes_per_number=2):
    return 2 * layers * d_model * n_words * bytes_per_number / 1024**3

models = {                       # layers, numbers per word
    "the paper's base model": (6, 512),
    "7B model (Llama 2)":     (32, 4096),
    "GPT-3 (175B)":           (96, 12288),
}
words_so_far = 32_000            # <- try 4_000 or 128_000

for name, (L, dm) in models.items():
    print(f"{name:24s} {kv_cache_gb(L, dm, words_so_far):8.2f} GB")

**Question 5** — A 7B model's weights take about 14 GB. Roughly how many words until its cache is bigger than the weights? What if 20 people use it at once?

*Your answer:*

---
## Quiz (Canvas): real interview questions

**Answer both**, in 2–3 sentences each, using words from the word bank.

1. "Create a KV cache in python." Describe what you would store and what happens at each step. — *Twinql AI, ML Engineer*
2. What is the time and memory cost of self-attention as the text gets longer? — *xAI, ML Engineer (part of a longer question)*

**Word bank:** KV cache · key · value · query · LayerNorm · Feed Forward · residual · mask · generation · $n^2$ · context window · memory

*Sources: candidate interview reports on Glassdoor, 2025.*